# AI Tutor — **Gemma multi-turn smoke** · 5 scenarios · single tab

The first OSS sweep zeroed all seven stock Gemmas before the models saw a
single prompt: Ollama's gemma templates have **no tool role**, so every
`tools` request returned HTTP 400 and every session deadlocked at turn 1.
This notebook smokes the **tool-enabled repackagings**
([okamototk/gemma3-tools](https://ollama.com/okamototk/gemma3-tools) — official
gemma3 weights, tool-enabled chat template) on **20 multi-turn
scenarios** per model, to answer *does Gemma run, pose, grade, and end
sessions cleanly through the engine?* before spending a full sweep on it.

- `okamototk/gemma3-tools:1b` — gemma3 1B
- `okamototk/gemma3-tools:4b` — gemma3 4B
- `okamototk/gemma3-tools:12b` — gemma3 12B
- `okamototk/gemma3-tools:27b` — gemma3 27B — largest; needs the big card

- **Student sim** = Anthropic **Haiku 4.5** persona player.
- **Judge** = Anthropic **Sonnet 4.6** @ temp 0 session rubric.
- **Engine** = `simple_tutor` with the full eval guard stack (the gemma tags
  resolve to the `gemma` family profile, so forcing/salvage/nets are active).
- **Cell 8c probes tool support first** — an incapable model is dropped in
  seconds, not 20 sessions.

> This is the **board run**: the same 20-scenario seed-5 draw as the
> engine fix-cycle board and the OSS qwen sweeps, so these numbers read directly
> against gemini 18/20, kimi 18/20, qwen3-next-80b 17/20, and the qwen OSS
> results. Gemma is evaluated here, separately from the qwen sweep, by design.
> Expectation from the smokes: 1b and 4b will score at the bottom (kept for the
> complete family record — comment them out of MODELS to save their ~40
> sessions of sim/judge cost if the record isn't needed).

**Runtime**: one tab covers all four models sequentially (weights are removed
after each model). **Pick an A100 (40 GB)** so the 27b fits.

**Before you start** — Colab Secrets (🔑 sidebar), each *Notebook access ON*:
- `GH_TOKEN` — GitHub classic PAT with `repo` scope (collaborator on `eai6/ai-tutor`).
- `ANTHROPIC_API_KEY` — **required** (student-sim + judge).
- `GOOGLE_API_KEY` + `OPENAI_API_KEY` — keep both so the cross-vendor grader
  cascade matches the other sweeps.

## Cell 1 — confirm GPU + mount Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — clone the repo (branch `pixeldesignlabs-dev-portuguese`)

In [ ]:
from google.colab import userdata
import subprocess, os
os.chdir('/content')   # a re-run's cwd may be the about-to-be-deleted clone
tok = (userdata.get('GH_TOKEN') or '').strip()
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'pixeldesignlabs-dev-portuguese', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())

## Cell 3 — fix hardcoded laptop paths (essential)

In [ ]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama (a few min; ignore pip resolver warnings)

In [ ]:
!pip install -q -r requirements.txt
# The Ollama installer is zstd-compressed; the Colab VM lacks zstd.
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
import subprocess, time, shutil, os
def _has_ollama():
    return shutil.which('ollama') is not None
def _install_ollama():
    if _has_ollama():
        return True
    # PINNED to 0.30.7 — the version the gemma3-tools tool templates were
    # verified against locally (5/5 probe passes). Colab's un-pinned latest
    # silently stopped parsing the repackaged template's tool calls: the same
    # model emitted zero tool_calls on every probe.
    for i in range(1, 4):
        print(f'[ollama] install.sh attempt {i} (pinned 0.30.7)', flush=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.30.7 sh', shell=True)
        if _has_ollama():
            return True
        time.sleep(5)
    for i in range(1, 4):
        print(f'[ollama] direct-binary attempt {i} (ollama.com, pinned)', flush=True)
        subprocess.run('curl -fL --retry 5 --retry-all-errors --connect-timeout 30 '
                       '-o /tmp/ollama.tgz "https://ollama.com/download/ollama-linux-amd64.tgz?version=0.30.7"',
                       shell=True)
        if os.path.exists('/tmp/ollama.tgz') and os.path.getsize('/tmp/ollama.tgz') > 1_000_000:
            subprocess.run('tar -C /usr -xzf /tmp/ollama.tgz', shell=True)
            if _has_ollama():
                return True
        time.sleep(5)
    return False
assert _install_ollama(), "ollama install failed — restart the runtime and retry"
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash', '-c', 'ollama list'], capture_output=True).returncode == 0:
        subprocess.run(['ollama', '--version']); print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

## Cell 5 — **required** — write .env from Colab Secrets

In [ ]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

## Cell 6 — fresh DB + eval fixtures

In [ ]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

## Cell 7 — persist results to Drive (symlink → survives disconnects)
Writes to `ai-tutor-eval-multiturn/gemma20_mt/`. Resume-safe: already-scored models are skipped on a rerun.

In [ ]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-multiturn/gemma20_mt
!rm -rf offline_eval/multi_turn_results/gemma20_mt && mkdir -p offline_eval/multi_turn_results && ln -s /content/drive/MyDrive/ai-tutor-eval-multiturn/gemma20_mt offline_eval/multi_turn_results/gemma20_mt
import os, glob
print('this run writes to:', os.path.realpath('offline_eval/multi_turn_results/gemma20_mt'))
done = sorted(os.path.basename(p)[:-5] for p in glob.glob('offline_eval/multi_turn_results/gemma20_mt/*.json'))
print('already scored:', done or '(none yet)')

## Cell 8 — write the model matrix + seed configs

In [ ]:
open('offline_eval/models.txt', 'w').write(
    "# Gemma multi-turn smoke — tool-enabled repackagings\n"
    "okamototk/gemma3-tools:1b  big     # gemma3 1B\nokamototk/gemma3-tools:4b  big     # gemma3 4B\nokamototk/gemma3-tools:12b big     # gemma3 12B\nokamototk/gemma3-tools:27b xl     # gemma3 27B — largest; needs the big card\n")
print(open('offline_eval/models.txt').read())
!python offline_eval/seed_ollama_configs.py
import django, os
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
django.setup()
from apps.llm.model_profiles import get_model_profile
for line in open('offline_eval/models.txt'):
    tag = line.split('#')[0].split()[0] if line.split('#')[0].split() else ''
    if not tag:
        continue
    p = get_model_profile(f'local_ollama/{tag}')
    assert p and p.family == 'gemma', f'{tag}: expected gemma family profile, got {p}'
    print(f'{tag:<26} family={p.family} mode={p.mode} sampling={p.sampling_dict()}')

## Cell 8b — reclaim disk BEFORE the run (harmless no-op on a fresh VM)

In [ ]:
!ollama list 2>/dev/null | tail -n +2 | awk '{print $1}' | xargs -r -n1 ollama rm 2>/dev/null || true
!rm -rf /root/.ollama/models/blobs/* /root/.ollama/models/manifests/* offline_eval/ollama_models/* 2>/dev/null || true
!pip cache purge 2>/dev/null || true
import subprocess
print(subprocess.run(['df','-h','/'],capture_output=True,text=True).stdout)

## Cell 8c — **tool-probe gate** (drops models that cannot tool-call)
Pulls each model, fires one probe tool-call, and REMOVES any model that fails from `models.txt` — a failure costs seconds, not sessions. This is the cell that would have caught the stock-Gemma 400s instantly.

In [ ]:
import subprocess, sys

kept, dropped = [], []
lines = open('offline_eval/models.txt').read().splitlines()
for line in lines:
    bare = line.split('#')[0].split()
    if not bare:
        continue
    tag = bare[0]
    print(f'>> probing {tag} ...', flush=True)
    pull = subprocess.run(['ollama', 'pull', tag], capture_output=True, text=True)
    if pull.returncode != 0:
        print(f'   PULL FAILED — dropped. ({pull.stderr.strip()[:120]})')
        dropped.append((tag, 'pull failed'))
        continue
    probe = subprocess.run(
        [sys.executable, 'offline_eval/_probe_ollama_tools.py', tag],
        capture_output=True, text=True)
    out = probe.stdout + probe.stderr
    if 'TOOL-USE SUPPORTED' in out:
        print('   tool-call OK')
        kept.append(line)
    else:
        reason = 'HTTP 400 (no tool role in template)' if '400' in out else 'no tool_calls emitted'
        print(f'   NO TOOL SUPPORT — dropped. ({reason})')
        dropped.append((tag, reason))
        subprocess.run(['ollama', 'rm', tag], capture_output=True)
    subprocess.run(['ollama', 'stop', tag], capture_output=True)

header = [l for l in lines if l.strip().startswith('#')]
open('offline_eval/models.txt', 'w').write('\n'.join(header + kept) + '\n')
print()
print('will sweep:', [l.split()[0] for l in kept] or 'NOTHING')
if dropped:
    print('dropped   :', dropped)
assert kept, 'Every model failed the tool probe — nothing to smoke.'

## Cell 9 — run the smoke (20 sessions per model; resume-safe)
**Sanity-check the first model before walking away:**
- `>> Tutor engine: simple_tutor` — not the legacy engine.
- The runner reports 20 scenarios.
- Sessions end `exit_ticket`/`completed`, not a wall of `deadlock`.
- Expect `pose salvaged` / `auto_grade_fallback` / `polarity_align` lines in the logs — that's the guard stack carrying a new family; a few is healthy, a storm is diagnostic gold either way.

In [ ]:
!RESULTS_DIR=$PWD/offline_eval/multi_turn_results/gemma20_mt SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 \
  MODE="--multi-turn --sample 20 --seed 5" bash offline_eval/run_matrix.sh

## Cell 10 — smoke board: pass + session end-reasons
n=20 per model — read the END-REASONS, not the pass rate. Clean = sessions reach `exit_ticket`/`completed`. `deadlock` at turn 1-2 = the template still isn't carrying tools; `max_turns` everywhere = pacing; `errored` = judge/sim infrastructure, re-run those.

In [ ]:
import json, glob, os
from collections import Counter
print(f"{'MODEL':<28}{'PASS':>7}   SESSION END-REASONS")
print('-' * 72)
for f in sorted(glob.glob('offline_eval/multi_turn_results/gemma20_mt/*.json')):
    name = os.path.basename(f)[:-5]
    if name.startswith('_'):
        continue
    d = json.load(open(f))
    ends = Counter(r.get('sim_reason') or ('errored' if r.get('error') else '?')
                   for r in d['results'])
    reasons = '  '.join(f"{k}:{v}" for k, v in ends.most_common())
    print(f"{name:<28}{d['passed']:>3}/{d['total_scenarios']:<3} {reasons}")
print()
print('Guard-stack activity per model (healthy = present, storm = look closer):')
for f in sorted(glob.glob('offline_eval/multi_turn_results/gemma20_mt/*.log')):
    name = os.path.basename(f)[:-4]
    log = open(f, errors='replace').read()
    counts = {k: log.count(k) for k in
              ('pose salvaged', 'auto_grade_fallback', 'polarity_align',
               'reveal_filter', 'dedupe_reply', 'call2_repair')}
    print(f"  {name:<26} " + '  '.join(f'{k.split()[0]}:{v}' for k, v in counts.items()))

## Reading the result

- **Clean smoke** (sessions end at exit_ticket, few guard interventions):
  promote Gemma into the full 20-scenario sweep — it's already wired into
  `colab_eval.ipynb`; just run that notebook's groups as usual.
- **Runs but messy** (heavy `polarity_align` / `auto_grade_fallback`, budget
  overruns): pull the transcripts from `MyDrive/ai-tutor-eval-multiturn/gemma20_mt/` into
  the repo's `offline_eval/multi_turn_results/gemma20_mt/` and do a bottleneck pass before the full sweep.
- **Still deadlocking at turn 1**: the repackaged template didn't hold —
  Cell 8c output says exactly why; fall back to option 2 (vLLM serving) from
  the Gemma analysis.

To pull results to the laptop: copy JSONs + logs from
`MyDrive/ai-tutor-eval-multiturn/gemma20_mt/` into `offline_eval/multi_turn_results/gemma20_mt/`.